# Ачча! — озвучка фраз туриста без акцента (Parler-TTS)

Этот блокнот озвучивает **только фразы туриста** открытой моделью **Parler-TTS Mini** (Hugging Face, лицензия Apache 2.0) — нейтральный американский английский, без индийского акцента. Реплики индийцев остаются из первого блокнота.

**Перед запуском:** меню **Среда выполнения → Сменить среду выполнения → T4 GPU → Сохранить**. Токен Hugging Face **не нужен**.

Запускайте ячейки по порядку кнопкой ▶. Всё вместе — 10–15 минут.

## 1. Установка (2–3 минуты)

В конце сеанс перезапустится сам и появится сообщение о сбое — это нормально. Дальше начинайте с шага 2.

In [ ]:
!pip -q install git+https://github.com/huggingface/parler-tts.git soundfile
!pip -q install -U "protobuf>=5.28"
!apt -qq install -y ffmpeg > /dev/null
print('Готово. Сеанс сейчас перезапустится сам — это нормально. После перезапуска начинайте с шага 2.')
import os; os.kill(os.getpid(), 9)

## 2. Настройки

- `VOICE` — голос. Женские: `Laura`, `Jenna`, `Lea`, `Emily`; мужские: `Jon`, `Gary`, `Mike`, `David`
- `FIX` — исправления произношения: слева как во фразе, справа как прочитать модели
- `ONLY_IDS` — переозвучить только эти фразы, например `['air-01']`; `[]` — все фразы туриста

In [ ]:
PHRASES_URL = 'https://achha-india.github.io/app/phrases.json'
VOICE = 'Laura'
STYLE = "{name} speaks slowly and clearly with a neutral American accent, in a calm friendly tone. The recording is very clear and close-up, with no background noise."
FIX = {
    'e-visa': 'e-veeza',
}
ONLY_IDS = []
OUT_DIR = '/content/drive/MyDrive/Achha/audio-tourist'
print('Настройки сохранены')

## 3. Модель и фразы (2–3 минуты)

In [ ]:
if 'PHRASES_URL' not in globals():
    raise RuntimeError('Сначала запустите шаг 2')
import json, urllib.request, os, subprocess, re, torch
import numpy as np, soundfile as sf
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', device, '(если cpu — включите T4 GPU, см. начало; на cpu тоже работает, но медленнее)')
MODEL_ID = 'parler-tts/parler-tts-mini-v1'
model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
SR = model.config.sampling_rate

data = json.load(urllib.request.urlopen(PHRASES_URL))
lines = [l for st in data['stages'] for s in st['situations'] for l in s['lines']]
todo = [l for l in lines if l['role'] == 'tourist']
if ONLY_IDS: todo = [l for l in lines if l['id'] in ONLY_IDS]
print('Фраз к озвучке:', len(todo))

def synth(text, seed=7):
    for k, v in FIX.items():
        text = re.sub(re.escape(k), v, text, flags=re.I)
    torch.manual_seed(seed)
    d = tokenizer(STYLE.format(name=VOICE), return_tensors='pt').to(device)
    p = tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
        g = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                           prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)
    a = g.cpu().numpy().squeeze().astype(np.float32)
    nz = np.where(np.abs(a) > 0.01)[0]
    if len(nz): a = a[max(0, nz[0] - int(0.05 * SR)): nz[-1] + int(0.15 * SR)]
    return a

def to_mp3(audio, path):
    wav = path.replace('.mp3', '.wav')
    sf.write(wav, audio, SR)
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', wav, '-ac', '1', '-b:a', '64k', path], check=True)
    os.remove(wav)
print('Готово')

## 4. Пробное прослушивание

Если голос не нравится — поменяйте `VOICE` на шаге 2 и запустите шаги 2 и 4 (шаг 3 повторять не нужно, но после смены `VOICE` запустите шаг 2).

In [ ]:
from IPython.display import Audio, display
if 'todo' not in globals():
    raise RuntimeError('Сначала запустите шаг 2 и шаг 3 — шаг 3 должен закончиться словом «Готово»')
for l in todo[:3]:
    print(l['id'], '·', l['en'])
    display(Audio(synth(l['en']), rate=SR))

## 5. Озвучить всё и сохранить в Google Drive

Colab попросит доступ к Google Drive — разрешите. Файлы появятся в папке **Achha/audio-tourist**, архив **achha-audio-tourist.zip** скачается сам. Его пришлите в чат.

In [ ]:
from google.colab import drive, files
import shutil
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
failed = []
for i, l in enumerate(todo, 1):
    path = f"{OUT_DIR}/{l['id']}.mp3"
    if os.path.exists(path) and not ONLY_IDS: continue
    try:
        to_mp3(synth(l['en']), path); print(f"[{i}/{len(todo)}] {l['id']} ✓")
    except Exception as e:
        failed.append(l['id']); print(f"[{i}/{len(todo)}] {l['id']} ошибка: {e}")
zip_path = shutil.make_archive('/content/drive/MyDrive/Achha/achha-audio-tourist', 'zip', OUT_DIR)
print('\nГотово | ошибки:', failed or 'нет')
files.download(zip_path)